In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

In [2]:
db_path = Path("../data/database.sqlite")

conn = sqlite3.connect(db_path.as_posix())

df_player = pd.read_sql_query("SELECT * FROM Player_Cleaned", conn)
df_player["date"] = pd.to_datetime(df_player["date"])

df_match = pd.read_sql_query("SELECT * FROM Match_Cleaned", conn)
df_match["date"] = pd.to_datetime(df_match["date"])

df_team = pd.read_sql_query("SELECT * FROM Team_Cleaned", conn)
df_team["date"] = pd.to_datetime(df_team["date"])

conn.close()

In [3]:
df_match = df_match.drop(
    columns=[
        "home_player_X1",
        "home_player_Y1",
        "away_player_X1",
        "away_player_Y1",
        "home_player_1",
        "away_player_1",
    ]
)

In [4]:
# merge_asof requires the "on" key (date) to be globally sorted
df_player_sorted = (
    df_player.dropna(subset=["player_api_id", "date"])
    .sort_values(["date", "player_api_id"], kind="mergesort")
    .reset_index(drop=True)
)

# Build a long version of df_match with one row per player in a match
player_cols = [f"home_player_{i}" for i in range(2, 12)] + [
    f"away_player_{i}" for i in range(2, 12)
]

df_match_long = (
    df_match[["match_api_id", "date"] + player_cols]
    .melt(
        id_vars=["match_api_id", "date"],
        value_vars=player_cols,
        var_name="position_col",
        value_name="player_api_id",
    )
    .dropna(subset=["player_api_id", "date"])
)

In [5]:
# player ids in df_match are float, cast them to int to match df_player
df_match_long["player_api_id"] = df_match_long["player_api_id"].astype("int64")

# Optional: keep side (home/away) and jersey number (2–11)
df_match_long["team_side"] = df_match_long["position_col"].str.split("_").str[0]
df_match_long["position_no"] = (
    df_match_long["position_col"].str.split("_").str[-1].astype("int64")
)

In [6]:
# Sort for merge_asof (date must be globally sorted)
df_match_long = df_match_long.sort_values(
    ["date", "player_api_id"], kind="mergesort"
).reset_index(drop=True)

In [7]:
# Pick for each (player, match_date) the player row
# whose date is closest to the match date
df_match_player = pd.merge_asof(
    df_match_long,
    df_player_sorted,
    by="player_api_id",
    left_on="date",
    right_on="date",
    direction="nearest",
)

In [8]:
df_match_player.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,player_name,height,weight
0,483137,2008-08-09,away_player_8,5217,away,8,99392,117096,72.0,79.0,...,47.0,36.00000,2.0,22.0,71.0,22.0,22.0,Laurent Batlles,175.26,150
1,483130,2008-08-09,away_player_2,11324,away,2,133215,50908,67.0,70.0,...,69.0,69.00000,11.0,20.0,58.0,20.0,20.0,Nicolas Seube,180.34,159
2,483130,2008-08-09,away_player_5,11327,away,5,14306,51121,69.0,73.0,...,54.0,55.00000,7.0,20.0,68.0,20.0,20.0,Anthony Deroin,162.56,146
3,483130,2008-08-09,away_player_8,11329,away,8,148156,142313,68.0,78.0,...,72.0,70.00000,6.0,23.0,65.0,23.0,23.0,Reynald Lemaitre,175.26,150
4,489982,2008-08-09,home_player_7,12985,home,7,30464,139894,63.0,69.0,...,63.0,61.49306,10.0,23.0,47.0,23.0,23.0,Christos Karypidis,185.42,176


In [9]:
# --- Add team features (closest to match date) for home/away teams ---

df_team_sorted = (
    df_team.dropna(subset=["team_api_id", "date"])
    .rename(columns={"date": "team_date"})
    .sort_values(["team_date", "team_api_id"], kind="mergesort")
    .reset_index(drop=True)
)

df_match_teams = df_match[
    [
        "match_api_id",
        "date",
        "home_team_api_id",
        "away_team_api_id",
        "home_team_goal",
        "away_team_goal",
    ]
].copy()

# Home team snapshot per match
home_left = (
    df_match_teams[["match_api_id", "date", "home_team_api_id"]]
    .rename(columns={"home_team_api_id": "team_api_id"})
    .dropna(subset=["team_api_id", "date"])
    .sort_values(["date", "team_api_id"], kind="mergesort")
    .reset_index(drop=True)
)

home_joined = pd.merge_asof(
    home_left,
    df_team_sorted,
    by="team_api_id",
    left_on="date",
    right_on="team_date",
    direction="nearest",
)

home_joined = home_joined.rename(columns={"team_api_id": "home_team_api_id"})
home_joined = home_joined.rename(
    columns={c: f"home_team_{c}" for c in df_team_sorted.columns if c != "team_api_id"}
)

# Away team snapshot per match
away_left = (
    df_match_teams[["match_api_id", "date", "away_team_api_id"]]
    .rename(columns={"away_team_api_id": "team_api_id"})
    .dropna(subset=["team_api_id", "date"])
    .sort_values(["date", "team_api_id"], kind="mergesort")
    .reset_index(drop=True)
)

away_joined = pd.merge_asof(
    away_left,
    df_team_sorted,
    by="team_api_id",
    left_on="date",
    right_on="team_date",
    direction="nearest",
)

away_joined = away_joined.rename(columns={"team_api_id": "away_team_api_id"})
away_joined = away_joined.rename(
    columns={c: f"away_team_{c}" for c in df_team_sorted.columns if c != "team_api_id"}
)

# One row per match with both home+away team features
# (keep match_api_id + date as join keys back into df_match_player)
df_match_team_features = df_match_teams.merge(
    home_joined, on=["match_api_id", "date", "home_team_api_id"], how="left"
).merge(away_joined, on=["match_api_id", "date", "away_team_api_id"], how="left")

# Final dataframe: one row per (match, player slot) with player + team features
# (df_match_player already contains match_api_id + date)
df_final = df_match_player.merge(
    df_match_team_features, on=["match_api_id", "date"], how="left"
)

In [10]:
df_final.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationPassing,away_team_chanceCreationCrossing,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name
0,483137,2008-08-09,away_player_8,5217,away,8,99392,117096,72.0,79.0,...,65.0,60.0,70.0,0.0,70.0,55.0,70.0,0.0,Grenoble Foot 38,GRE
1,483130,2008-08-09,away_player_2,11324,away,2,133215,50908,67.0,70.0,...,50.0,65.0,40.0,0.0,70.0,65.0,70.0,0.0,SM Caen,CAE
2,483130,2008-08-09,away_player_5,11327,away,5,14306,51121,69.0,73.0,...,50.0,65.0,40.0,0.0,70.0,65.0,70.0,0.0,SM Caen,CAE
3,483130,2008-08-09,away_player_8,11329,away,8,148156,142313,68.0,78.0,...,50.0,65.0,40.0,0.0,70.0,65.0,70.0,0.0,SM Caen,CAE
4,489982,2008-08-09,home_player_7,12985,home,7,30464,139894,63.0,69.0,...,70.0,70.0,65.0,0.0,60.0,70.0,70.0,0.0,Motherwell,MOT


In [11]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 427220 entries, 0 to 427219
Data columns (total 87 columns):
 #   Column                                    Non-Null Count   Dtype         
---  ------                                    --------------   -----         
 0   match_api_id                              427220 non-null  int64         
 1   date                                      427220 non-null  datetime64[us]
 2   position_col                              427220 non-null  str           
 3   player_api_id                             427220 non-null  int64         
 4   team_side                                 427220 non-null  object        
 5   position_no                               427220 non-null  int64         
 6   id                                        427220 non-null  int64         
 7   player_fifa_api_id                        427220 non-null  int64         
 8   overall_rating                            427220 non-null  float64       
 9   potential                 

In [12]:
df_final[
    df_final["home_team_team_long_name"].str.contains("Bayern")
    | df_final["away_team_team_long_name"].str.contains("Bayern")
]

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationPassing,away_team_chanceCreationCrossing,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name
200,499317,2008-08-15,away_player_3,27293,away,3,19604,11114,74.0,78.0,...,70.0,60.0,70.0,0.0,70.0,45.0,60.0,0.0,Hamburger SV,HAM
201,499317,2008-08-15,away_player_7,30598,away,7,133420,53578,80.0,87.0,...,70.0,60.0,70.0,0.0,70.0,45.0,60.0,0.0,Hamburger SV,HAM
202,499317,2008-08-15,home_player_10,30638,home,10,127309,11141,85.0,90.0,...,70.0,60.0,70.0,0.0,70.0,45.0,60.0,0.0,Hamburger SV,HAM
203,499317,2008-08-15,away_player_6,30749,away,6,39742,5345,77.0,81.0,...,70.0,60.0,70.0,0.0,70.0,45.0,60.0,0.0,Hamburger SV,HAM
204,499317,2008-08-15,away_player_11,30764,away,11,74526,148138,79.0,85.0,...,70.0,60.0,70.0,0.0,70.0,45.0,60.0,0.0,Hamburger SV,HAM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426361,2002487,2016-05-14,away_player_10,375732,away,10,74443,213777,70.0,77.0,...,38.0,51.0,40.0,0.0,35.0,50.0,61.0,0.0,Hannover 96,HAN
426380,2002487,2016-05-14,home_player_7,429265,home,7,97108,213345,78.0,86.0,...,38.0,51.0,40.0,0.0,35.0,50.0,61.0,0.0,Hannover 96,HAN
426407,2002487,2016-05-14,away_player_8,501569,away,8,176033,220881,62.0,75.0,...,38.0,51.0,40.0,0.0,35.0,50.0,61.0,0.0,Hannover 96,HAN
426444,2002487,2016-05-14,away_player_3,664970,away,3,178541,229476,58.0,73.0,...,38.0,51.0,40.0,0.0,35.0,50.0,61.0,0.0,Hannover 96,HAN


In [13]:
df_final[df_final["match_api_id"] == 499317]["player_name"]

200         Bastian Reinhardt
201             Nigel de Jong
202            Miroslav Klose
203             David Jarolim
204                Ivica Olic
205    Bastian Schweinsteiger
206              Philipp Lahm
207            Lukas Podolski
208         Daniel van Buyten
209       Jose Paolo Guerrero
210            Christian Lell
211            Jerome Boateng
212           Collin Benjamin
213           Joris Mathijsen
214         Jonathan Pitroipa
215           Mark van Bommel
216                Ze Roberto
217          Piotr Trochowski
218                     Lucio
219                Toni Kroos
Name: player_name, dtype: str

In [14]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 427220 entries, 0 to 427219
Data columns (total 87 columns):
 #   Column                                    Non-Null Count   Dtype         
---  ------                                    --------------   -----         
 0   match_api_id                              427220 non-null  int64         
 1   date                                      427220 non-null  datetime64[us]
 2   position_col                              427220 non-null  str           
 3   player_api_id                             427220 non-null  int64         
 4   team_side                                 427220 non-null  object        
 5   position_no                               427220 non-null  int64         
 6   id                                        427220 non-null  int64         
 7   player_fifa_api_id                        427220 non-null  int64         
 8   overall_rating                            427220 non-null  float64       
 9   potential                 

In [15]:
# --- Add player-level X/Y coordinates for each match ---

coord_rows = []
for side in ["home", "away"]:
    for i in range(2, 12):
        x_col = f"{side}_player_X{i}"
        y_col = f"{side}_player_Y{i}"
        tmp = df_match[["match_api_id", "date", x_col, y_col]].copy()
        tmp["team_side"] = side
        tmp["position_no"] = i
        tmp = tmp.rename(columns={x_col: "player_x", y_col: "player_y"})
        coord_rows.append(tmp)

# One row per (match, team_side, position_no) with X/Y
df_match_coords = pd.concat(coord_rows, ignore_index=True)

# Attach coordinates to the final player-level dataframe
# Keys are match_api_id + date + team_side + position_no (all already in df_final)
df_final = df_final.merge(
    df_match_coords,
    on=["match_api_id", "date", "team_side", "position_no"],
    how="left",
)

In [17]:
df_final = df_final.drop(
    columns=[
        "gk_diving",
        "gk_handling",
        "gk_kicking",
        "gk_positioning",
        "gk_reflexes",
    ]
)

In [18]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 427220 entries, 0 to 427219
Data columns (total 84 columns):
 #   Column                                    Non-Null Count   Dtype         
---  ------                                    --------------   -----         
 0   match_api_id                              427220 non-null  int64         
 1   date                                      427220 non-null  datetime64[us]
 2   position_col                              427220 non-null  str           
 3   player_api_id                             427220 non-null  int64         
 4   team_side                                 427220 non-null  object        
 5   position_no                               427220 non-null  int64         
 6   id                                        427220 non-null  int64         
 7   player_fifa_api_id                        427220 non-null  int64         
 8   overall_rating                            427220 non-null  float64       
 9   potential                 